In [0]:
%pip install dbtunnel[gradio] aiohttp

In [0]:
dbutils.library.restartPython()

In [0]:
import os
from databricks.sdk import WorkspaceClient

# Use notebook built-in auth instead of secret token (same approach as rag_chatbot)
_w = WorkspaceClient()
os.environ['DATABRICKS_TOKEN'] = _w.config.authenticate()["Authorization"].split(" ", 1)[1]

os.environ['API_ENDPOINT'] = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/serving-endpoints/databricks-meta-llama-3-3-70b-instruct/invocations"


In [0]:
import os
import itertools
import gradio as gr
import requests
import json

from databricks.sdk import WorkspaceClient
from gradio.themes.utils import sizes

# Use notebook built-in auth (same approach as the previous cells)
_w = WorkspaceClient()


def respond(message, history):

    if len(message.strip()) == 0:
        return "ERROR the question should not be empty"
    
    print ("#### Message #####")
    print (message)

    local_token = _w.config.authenticate()["Authorization"].split(" ", 1)[1]
    local_endpoint = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/serving-endpoints/databricks-meta-llama-3-3-70b-instruct/invocations"

    # Add your API token to the headers
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {local_token}'
    }

    payload = {
        "inputs" : [message]

    }

    try:
        response = requests.post(
            local_endpoint, json=payload, headers=headers, timeout=100)

        response_json = response.json()
        print ("#### RESPONSE JSON")
        print (response_json)

        response_data = response_json["predictions"][0]  # Updated line to extract the result field
        print(response_data)

    except Exception as error:
        response_data = f"ERROR status_code: {type(error).__name__}"
        # + str(response.status_code) + " response:" + response.text

    # print(response.json())
    return response_data


theme = gr.themes.Soft(
    text_size=sizes.text_sm,radius_size=sizes.radius_sm, spacing_size=sizes.spacing_sm,
)


demo = gr.ChatInterface(
    respond,
    chatbot=gr.Chatbot(show_label=False, container=False, show_copy_button=True, bubble_full_width=True),
    textbox=gr.Textbox(placeholder="Ask me a question",
                       container=False, scale=7),
    title="LH Nautical Chatbot",
    description="Esse Chatbot responde a perguntas de negócio sobre a empresa LH Nautical.",
    examples=[["Há necessidade de construção de mais centros de distribuição? Se sim, onde?"],],
    cache_examples=False,
    theme=theme,
    retry_btn=None,
    undo_btn=None,
    clear_btn="Clear",
)

# dbtunnel requires JVM access (not available on serverless).
# Gradio share=True needs frpc binary (download blocked on serverless).
# Using inline=True to render directly in notebook output.
demo.launch(inline=True, share=False)